# AWS Cost Analysis & Optimization

**Category:** 09-FinOps-Cloud  
**Description:** Analyze AWS costs and get AI-powered optimization recommendations  
**Requirements:** AWS credentials with Cost Explorer access

---

## What You'll Learn

1. Query AWS Cost Explorer
2. Visualize spending trends
3. Identify cost anomalies
4. Get AI optimization recommendations
5. Build cost dashboards

---

In [ ]:
# =============================================================================
# DEPENDENCIES & MODEL ALIASES
# =============================================================================
%pip install -q boto3 pandas matplotlib seaborn

# Model aliases
CLAUDE = "anthropic-chat:claude-sonnet-4-5-20250929"
GPT = "openai-chat:gpt-5"
GEMINI = "gemini:gemini-2.5-flash"

MODEL = CLAUDE

In [ ]:
import boto3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')

---

# Part 1: AWS Cost Explorer Setup

---

In [ ]:
# Initialize Cost Explorer client
# Uses AWS credentials from environment or ~/.aws/credentials
try:
    ce = boto3.client('ce', region_name='us-east-1')
    print("Cost Explorer client initialized successfully")
except Exception as e:
    print(f"Error: {e}")
    print("\nMake sure AWS credentials are configured with Cost Explorer access.")

In [ ]:
# Date range for analysis
end_date = datetime.now().strftime('%Y-%m-%d')
start_date = (datetime.now() - timedelta(days=90)).strftime('%Y-%m-%d')

print(f"Analysis period: {start_date} to {end_date}")

---

# Part 2: Cost by Service

---

In [ ]:
def get_cost_by_service(start, end):
    """Get cost breakdown by AWS service."""
    try:
        response = ce.get_cost_and_usage(
            TimePeriod={'Start': start, 'End': end},
            Granularity='MONTHLY',
            Metrics=['UnblendedCost'],
            GroupBy=[{'Type': 'DIMENSION', 'Key': 'SERVICE'}]
        )
        
        # Parse results
        data = []
        for result in response['ResultsByTime']:
            period = result['TimePeriod']['Start']
            for group in result['Groups']:
                service = group['Keys'][0]
                cost = float(group['Metrics']['UnblendedCost']['Amount'])
                data.append({'period': period, 'service': service, 'cost': cost})
        
        return pd.DataFrame(data)
    
    except Exception as e:
        print(f"Error fetching costs: {e}")
        # Return sample data for demo
        return create_sample_cost_data()

def create_sample_cost_data():
    """Create sample data for demonstration."""
    services = ['Amazon EC2', 'Amazon S3', 'Amazon RDS', 'AWS Lambda', 
                'Amazon CloudFront', 'Amazon DynamoDB', 'Others']
    periods = pd.date_range(start=start_date, end=end_date, freq='MS').strftime('%Y-%m-%d').tolist()
    
    data = []
    base_costs = [1500, 800, 1200, 300, 200, 400, 500]
    
    for period in periods:
        for service, base in zip(services, base_costs):
            # Add some randomness
            cost = base * (1 + np.random.uniform(-0.2, 0.3))
            data.append({'period': period, 'service': service, 'cost': cost})
    
    return pd.DataFrame(data)

In [ ]:
# Get cost data
costs_df = get_cost_by_service(start_date, end_date)

print(f"Retrieved {len(costs_df)} cost records")
costs_df.head(10)

In [ ]:
# Aggregate by service
service_totals = costs_df.groupby('service')['cost'].sum().sort_values(ascending=False)

print("Total Cost by Service:")
for service, cost in service_totals.items():
    print(f"  {service}: ${cost:,.2f}")

print(f"\nTotal: ${service_totals.sum():,.2f}")

In [ ]:
# Visualize cost by service
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
top_services = service_totals.head(7)
axes[0].barh(top_services.index, top_services.values, color='steelblue')
axes[0].set_xlabel('Cost ($)')
axes[0].set_title('Cost by AWS Service')
axes[0].invert_yaxis()

# Pie chart
axes[1].pie(top_services.values, labels=top_services.index, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Cost Distribution')

plt.tight_layout()
plt.show()

---

# Part 3: Cost Trends Over Time

---

In [ ]:
# Pivot for time series
costs_pivot = costs_df.pivot(index='period', columns='service', values='cost')
costs_pivot.index = pd.to_datetime(costs_pivot.index)

# Monthly totals
monthly_totals = costs_df.groupby('period')['cost'].sum()
monthly_totals.index = pd.to_datetime(monthly_totals.index)

# Plot trends
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Total monthly cost
axes[0].bar(monthly_totals.index, monthly_totals.values, color='steelblue', width=20)
axes[0].plot(monthly_totals.index, monthly_totals.values, 'ro-', markersize=8)
axes[0].set_title('Total Monthly AWS Cost', fontsize=14)
axes[0].set_ylabel('Cost ($)')

# Add trend line
z = np.polyfit(range(len(monthly_totals)), monthly_totals.values, 1)
p = np.poly1d(z)
axes[0].plot(monthly_totals.index, p(range(len(monthly_totals))), 'g--', 
             label=f'Trend (${z[0]:+.0f}/month)')
axes[0].legend()

# Stacked area by service
top_5 = costs_df.groupby('service')['cost'].sum().nlargest(5).index
costs_top5 = costs_pivot[top_5]
costs_top5.plot.area(ax=axes[1], stacked=True, alpha=0.7)
axes[1].set_title('Cost Breakdown by Top Services', fontsize=14)
axes[1].set_ylabel('Cost ($)')
axes[1].legend(loc='upper left', bbox_to_anchor=(1, 1))

plt.tight_layout()
plt.show()

In [ ]:
# Month-over-month change
mom_change = monthly_totals.pct_change() * 100

fig, ax = plt.subplots(figsize=(12, 5))

colors = ['green' if x < 0 else 'red' for x in mom_change.values]
ax.bar(mom_change.index, mom_change.values, color=colors, width=20)
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

ax.set_title('Month-over-Month Cost Change (%)', fontsize=14)
ax.set_ylabel('Change (%)')
ax.set_xlabel('Month')

plt.tight_layout()
plt.show()

---

# Part 4: Cost Anomaly Detection

---

In [ ]:
# Simple anomaly detection using z-scores
from scipy import stats

def detect_anomalies(series, threshold=2):
    """Detect anomalies using z-score method."""
    z_scores = np.abs(stats.zscore(series))
    anomalies = z_scores > threshold
    return anomalies

# Check each service for anomalies
print("Cost Anomaly Detection:")
print("=" * 50)

anomaly_services = []
for service in costs_pivot.columns:
    service_costs = costs_pivot[service].dropna()
    if len(service_costs) >= 3:
        anomalies = detect_anomalies(service_costs, threshold=1.5)
        if anomalies.any():
            anomaly_dates = service_costs[anomalies].index
            anomaly_values = service_costs[anomalies].values
            avg_cost = service_costs.mean()
            
            for date, value in zip(anomaly_dates, anomaly_values):
                pct_diff = ((value - avg_cost) / avg_cost) * 100
                print(f"\n{service}:")
                print(f"  Date: {date.strftime('%Y-%m')}")
                print(f"  Cost: ${value:,.2f} (Avg: ${avg_cost:,.2f})")
                print(f"  Deviation: {pct_diff:+.1f}%")
                anomaly_services.append(service)

---

# Part 5: AI-Powered Optimization Recommendations

---

In [ ]:
# Prepare cost summary for AI analysis
cost_summary = f"""
AWS Cost Analysis Summary (Last 90 Days)
=========================================

Total Spending: ${service_totals.sum():,.2f}

Top Services by Cost:
"""

for i, (service, cost) in enumerate(service_totals.head(7).items(), 1):
    pct = (cost / service_totals.sum()) * 100
    cost_summary += f"  {i}. {service}: ${cost:,.2f} ({pct:.1f}%)\n"

cost_summary += f"""
Monthly Trend: ${z[0]:+.0f}/month

Anomalies Detected:
  Services with unusual spending: {len(set(anomaly_services))}
"""

print(cost_summary)

In [ ]:
%%ai $MODEL
Analyze this AWS cost report and provide specific optimization recommendations:

AWS Cost Analysis Summary (Last 90 Days):
- Total: ~$15,000
- EC2: $4,500 (30%) 
- RDS: $3,600 (24%)
- S3: $2,400 (16%)
- Others: $4,500 (30%)

Trend: Costs increasing ~$200/month

Please provide:
1. Top 5 specific cost optimization recommendations
2. Estimated savings for each recommendation
3. Implementation difficulty (Easy/Medium/Hard)
4. Quick wins vs long-term optimizations
5. Any red flags or concerns about the spending pattern

---

# Part 6: Reserved Instance Analysis

---

In [ ]:
def get_ri_recommendations():
    """Get Reserved Instance recommendations."""
    try:
        response = ce.get_reservation_purchase_recommendation(
            Service='Amazon Elastic Compute Cloud - Compute',
            TermInYears='ONE_YEAR',
            PaymentOption='NO_UPFRONT'
        )
        return response.get('Recommendations', [])
    except Exception as e:
        print(f"Note: {e}")
        return []

ri_recommendations = get_ri_recommendations()

if ri_recommendations:
    print("Reserved Instance Recommendations:")
    for rec in ri_recommendations:
        print(f"  Estimated Monthly Savings: ${rec.get('EstimatedMonthlySavingsAmount', 'N/A')}")
else:
    print("No RI recommendations available (or insufficient permissions)")
    print("\nTypical RI savings: 30-50% compared to On-Demand pricing")

---

# Part 7: Cost Dashboard

---

In [ ]:
# Create comprehensive cost dashboard
fig = plt.figure(figsize=(16, 12))

# Grid layout
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# 1. Total cost KPI
ax1 = fig.add_subplot(gs[0, 0])
ax1.text(0.5, 0.6, f"${service_totals.sum():,.0f}", fontsize=32, ha='center', fontweight='bold')
ax1.text(0.5, 0.3, 'Total (90 days)', fontsize=14, ha='center', color='gray')
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 1)
ax1.axis('off')
ax1.set_title('Total AWS Spend', fontsize=14, fontweight='bold')

# 2. Monthly average KPI
ax2 = fig.add_subplot(gs[0, 1])
monthly_avg = monthly_totals.mean()
ax2.text(0.5, 0.6, f"${monthly_avg:,.0f}", fontsize=32, ha='center', fontweight='bold', color='steelblue')
ax2.text(0.5, 0.3, 'Monthly Average', fontsize=14, ha='center', color='gray')
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)
ax2.axis('off')
ax2.set_title('Avg Monthly Cost', fontsize=14, fontweight='bold')

# 3. Trend KPI
ax3 = fig.add_subplot(gs[0, 2])
trend_color = 'red' if z[0] > 0 else 'green'
ax3.text(0.5, 0.6, f"${z[0]:+,.0f}", fontsize=32, ha='center', fontweight='bold', color=trend_color)
ax3.text(0.5, 0.3, 'Monthly Trend', fontsize=14, ha='center', color='gray')
ax3.set_xlim(0, 1)
ax3.set_ylim(0, 1)
ax3.axis('off')
ax3.set_title('Cost Trend', fontsize=14, fontweight='bold')

# 4. Monthly trend (spans 2 columns)
ax4 = fig.add_subplot(gs[1, :2])
ax4.bar(monthly_totals.index, monthly_totals.values, color='steelblue', width=20)
ax4.plot(monthly_totals.index, p(range(len(monthly_totals))), 'r--', linewidth=2)
ax4.set_title('Monthly Cost Trend', fontweight='bold')
ax4.set_ylabel('Cost ($)')

# 5. Service breakdown pie
ax5 = fig.add_subplot(gs[1, 2])
ax5.pie(top_services.values, labels=None, autopct='%1.0f%%', startangle=90)
ax5.set_title('By Service', fontweight='bold')
ax5.legend(top_services.index, loc='center left', bbox_to_anchor=(-0.3, 0.5), fontsize=8)

# 6. Top services bar (spans full width)
ax6 = fig.add_subplot(gs[2, :])
colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(top_services)))
bars = ax6.barh(top_services.index, top_services.values, color=colors)
ax6.set_xlabel('Cost ($)')
ax6.set_title('Top Services by Cost', fontweight='bold')
ax6.invert_yaxis()

# Add value labels
for bar, value in zip(bars, top_services.values):
    ax6.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2, 
             f'${value:,.0f}', va='center')

fig.suptitle('AWS Cost Dashboard', fontsize=18, fontweight='bold', y=0.98)

plt.savefig('aws_cost_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

print("Dashboard saved to: aws_cost_dashboard.png")

---

## Summary

This notebook demonstrated:

1. **AWS Cost Explorer** - Querying cost and usage data
2. **Service Breakdown** - Analyzing costs by AWS service
3. **Trend Analysis** - Monthly cost trends and projections
4. **Anomaly Detection** - Identifying unusual spending
5. **AI Recommendations** - Getting optimization suggestions
6. **RI Analysis** - Reserved Instance opportunities
7. **Dashboards** - Building cost visibility tools

---

**Next:** Try resource utilization analysis in `09-FinOps-Cloud/02-resource-optimization.ipynb`